# Stage 3 deep — QTIP 3-bit DECISIVE quality gate (f16-source, vs Q4_K_M, on code)

**State: RUNNABLE — this IS the decisive gate.** The local CPU proxy
(`tools/bench/oracle_qtip_quality.py --local-proxy`, committed; report
`reports/oracle_qtip_quality.md`) already ran and returned the one number that
sends us here: at a ~3.0-bit byte budget QTIP must extract **~1.20 bits** of
combined RHT-whitening + trellis coding gain over a 3-bit scalar just to MATCH
Q4_K_M's weight-RMSE — the **upper edge** of the TCQ envelope, and the modeled
+1-bit upper bound still fell ~20% short on the bootstrap. A CPU proxy on
resampled Q4_K-dequant values cannot legitimately KILL the method (that would
be a Type-2 error per CLAUDE.md): the decisive test needs the **real f16 Qwen**,
the **real QTIP RHT+trellis codec fit from f16**, and a forward pass. None of
those exist on the M3. This notebook is that decisive Colab run.

## The EXACT pass/fail criterion (baked into the verdict cell)

QTIP-3.0 (fit from f16) is judged against the **real Q4_K_M GGUF** (the shipped
incumbent it would replace) on three legs, on a **code** corpus:

1. **Weight-RMSE / the ~1.20-bit line (cheap first cut).** Per-tensor relative
   RMSE of the QTIP decode vs f16, and of the real Q4_K_M decode vs f16. The
   proxy showed QTIP needs ~1.20 bits of coding gain to close the RMSE gap; on
   real f16 the actual `bits_needed = log2(rmse_qtip / rmse_q4km)` is measured
   directly. **GO floor:** median QTIP-3.0 RMSE ≤ Q4_K_M RMSE (i.e. measured
   `bits_needed ≤ 0`) at FEWER bytes (~96–104 vs 144 weight B/256-blk). The proxy
   said this is exactly where the trellis lives or dies.
2. **Logit divergence vs Q4_K_M on code (the decisive functional leg).**
   logit-cosine, KL(f16∥·), and greedy argmax-agreement of next-token logits on
   held-out code tokens. **GO floor:** QTIP ≥ Q4_K_M cosine AND ≥ Q4_K_M argmax
   AND ≤ Q4_K_M KL (QTIP must be at least as good as the incumbent it replaces).
3. **Code PPL (corroborating, llama.cpp gold Q4_K_M).** QTIP-3.0 PPL ≤ 1.10× f16
   PPL and ≤ Q4_K_M PPL on the code eval set.

**Honest-verdict rule (mirrors the oracle).** Every number is tagged
(measured)/(proxy). The decode codec has two forms in this notebook — the REAL
upstream QTIP RHT+bitshift-trellis (decisive), and the oracle's FAITHFUL BRACKET
(RHT + Lloyd-Max {K, K+1}-bit scalar at the same stored ~K bits) as a
VRAM-cheap fallback if the QTIP CUDA kernels do not build. The bracket is
explicitly labelled and gives [lower, upper] — a NO-GO is only recorded from the
REAL codec; if only the bracket ran, the verdict is NEEDS-MEASUREMENT and names
the gate (real-QTIP rebuild).

**Runtime:** ~30–60 min on an L4/A100 (Hessian + quant is the heavy step).
GPU REQUIRED (f16 forward pass + QTIP quant). Produces `qtip_3bit_results.json`.

In [ ]:
# --- 0. Config + GPU preflight + the baked-in decision constants -------------
# This notebook is RUNNABLE end-to-end. The only knob is RUN_REAL_QTIP: when
# True we fit the upstream Cornell-RelaxML/QTIP codec from f16 (the decisive
# codec); the oracle's faithful RHT+Lloyd-Max bracket ALWAYS runs as a cheap
# cross-check / fallback so the gate fires even if the QTIP CUDA kernels fail to
# build on the assigned GPU.
import json, os, subprocess, sys, time, re
from pathlib import Path

import torch
assert torch.cuda.is_available(), "GPU REQUIRED: f16 forward pass + QTIP quant."
_gpu = torch.cuda.get_device_name(0)
_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print("GPU:", _gpu, f"{_vram:.0f} GB")

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"          # f16 source (the only fair input)
BITS = 3                                         # QTIP stored bits/weight target
RUN_REAL_QTIP = _vram >= 24                      # need L4/A100-class for QTIP quant

# ---- The EXACT pass/fail lines (from reports/oracle_qtip_quality.md) ---------
# The proxy's headline: ~1.20 bits of RHT+trellis coding gain over a 3-bit
# scalar are needed to MATCH Q4_K_M weight-RMSE. On real f16 we measure the
# actual bits_needed; GO requires it to reach <= 0 (QTIP RMSE <= Q4_K_M RMSE).
PROXY_BITS_NEEDED = 1.20            # what the CPU proxy estimated (for context only)
GATE_BITS_NEEDED = 0.0             # GO floor: measured bits_needed must reach <= 0
GATE_PPL_RATIO = 1.10             # QTIP PPL <= 1.10x f16 PPL
# Logit-leg floors are RELATIVE to the real Q4_K_M baseline (computed at runtime):
#   GO: cos(QTIP,f16) >= cos(Q4K,f16) AND argmax(QTIP,f16) >= argmax(Q4K,f16)
#       AND KL(f16||QTIP) <= KL(f16||Q4K).

RHT_BLOCK = 256                    # incoherence rotation block (matches oracle)
QK = 256                           # ggml K-quant super-block
SEED = 0

results = {
    "model": MODEL_ID, "bits": BITS, "gpu": _gpu, "vram_gb": round(_vram, 1),
    "run_real_qtip": RUN_REAL_QTIP,
    "proxy_bits_needed": PROXY_BITS_NEEDED,
    "gate": {"bits_needed_le": GATE_BITS_NEEDED, "ppl_ratio_le": GATE_PPL_RATIO,
             "logit": "cos>=Q4K & argmax>=Q4K & KL<=Q4K, on code"},
    "verdict": "PENDING",
}
print("RUN_REAL_QTIP =", RUN_REAL_QTIP,
      "(real upstream QTIP codec)" if RUN_REAL_QTIP else
      "(VRAM<24GB -> faithful BRACKET only; verdict will be NEEDS-MEASUREMENT)")
print(f"Decision lines: bits_needed <= {GATE_BITS_NEEDED} (proxy estimated",
      f"~{PROXY_BITS_NEEDED}); PPL ratio <= {GATE_PPL_RATIO}; logit vs Q4_K_M.")

In [ ]:
# --- 1. Install deps + build llama.cpp (for the GOLD Q4_K_M reference) -------
# gguf-python has NO K-quant quantizer (verified NotImplementedError for Q4_K),
# so the fair Q4_K_M baseline is the SHIPPED llama.cpp quantizer, not a NumPy
# reimplementation. We build it once; it gives us (a) the Q4_K_M .gguf to
# dequantize for the RMSE leg and (b) llama-perplexity for the PPL leg.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.44", "accelerate", "safetensors",
                "huggingface_hub", "gguf", "scipy"], check=False)
import numpy as np

REPO = "/content/llama.cpp"
BIN = f"{REPO}/build/bin"
if not Path(BIN, "llama-quantize").exists():
    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ggml-org/llama.cpp", REPO], check=True)
    flags = ["-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release", "-DGGML_CUDA=ON"]
    subprocess.run(["cmake", "-S", REPO, "-B", f"{REPO}/build", *flags], check=True)
    subprocess.run(["cmake", "--build", f"{REPO}/build", "-j", "--config", "Release"],
                   check=True)
for tool in ("llama-quantize", "llama-perplexity"):
    assert Path(BIN, tool).exists(), f"build missing {tool}"
print("llama.cpp built:", BIN)

In [ ]:
# --- 2. Fetch the f16 Qwen (HF + a near-lossless GGUF) + the code corpora ----
# f16 source serves two roles: (a) the HF model is the forward-pass reference
# and the QTIP quant input; (b) a near-lossless GGUF (q8_0/f16) is the
# llama.cpp source we quantize to the GOLD Q4_K_M (NOT requant-from-Q4 -- the
# recorded kill-respect: the byte-cut prize needs from-f16, never from-Q4_K).
from huggingface_hub import list_repo_files, hf_hub_download
import urllib.request

REPOS = ["Qwen/Qwen2.5-3B-Instruct-GGUF", "bartowski/Qwen2.5-3B-Instruct-GGUF"]
PREF = ["q8_0", "f16", "fp16"]                   # near-lossless = the f16 stand-in
def find_source():
    for repo in REPOS:
        try:
            ggufs = [f for f in list_repo_files(repo) if f.lower().endswith(".gguf")]
        except Exception as e:
            print("skip", repo, e); continue
        single = [f for f in ggufs if "-of-" not in f]
        for pref in PREF:
            hit = sorted((f for f in single if pref in f.lower()), key=len)
            if hit:
                return repo, hit[0]
    return None, None
gg_repo, gg_name = find_source()
assert gg_name, f"no single-file q8_0/f16 GGUF in {REPOS}; edit REPOS"
SRC_GGUF = hf_hub_download(gg_repo, gg_name)
results["source_gguf"] = f"{gg_repo}/{gg_name}"
print("near-lossless source GGUF:", SRC_GGUF)

# Gold Q4_K_M, quantized from the near-lossless source (from-f16-class, NOT
# from-an-already-Q4 file). --allow-requantize only matters if source is q8_0;
# q8_0 -> Q4_K_M is the standard llama.cpp recipe and is the fair incumbent.
Q4KM_GGUF = "/content/qwen3b-q4_k_m.gguf"
if not Path(Q4KM_GGUF).exists():
    r = subprocess.run([f"{BIN}/llama-quantize", "--allow-requantize",
                        SRC_GGUF, Q4KM_GGUF, "Q4_K_M"], capture_output=True, text=True)
    assert Path(Q4KM_GGUF).exists(), r.stderr[-1000:]
print("gold Q4_K_M:", Q4KM_GGUF,
      f"({os.path.getsize(Q4KM_GGUF)/1024**3:.2f} GiB)")

# Code corpora (same files the local oracle / nb01 use).
RAW = ("https://raw.githubusercontent.com/joshuahickscorp/dismantle/"
       "codex/maximal-spec-colab/colab/data")
for name in ("calib_trim.txt", "ppl_trim.txt"):
    if not Path("/content", name).exists():
        urllib.request.urlretrieve(f"{RAW}/{name}", f"/content/{name}")
CALIB, EVAL = "/content/calib_trim.txt", "/content/ppl_trim.txt"
print("code corpora ready:", CALIB, EVAL)

In [ ]:
# --- 3a. The codec models: RHT (exact) + faithful Lloyd-Max bracket ----------
# This block is the SAME codec model as tools/bench/oracle_qtip_quality.py, so
# the Colab bracket leg is directly comparable to the committed local proxy.
# RHT is modeled EXACTLY (orthogonal). The trellis coding gain is BRACKETED by
# running RHT + an optimal scalar quantizer at nlevels = 2^K (lower bound, no
# trellis gain) and 2^(K+1) (upper bound, full ~1-bit gain), both at the same
# stored ~K-bit byte budget. 'A wrong trellis sim is worse than none' -- the
# bracket refuses to invent a single trellis number; the REAL codec (3b) does.
import numpy as np

def _fwht(a):
    a = a.astype(np.float64); orig = a.shape; n = orig[-1]
    a = a.reshape(-1, n).copy(); h = 1
    while h < n:
        a = a.reshape(-1, n // (2 * h), 2, h)
        x = a[:, :, 0, :]; y = a[:, :, 1, :]
        a = np.concatenate([x + y, x - y], axis=-1).reshape(-1, n); h *= 2
    return (a / np.sqrt(n)).reshape(orig)

def _rht_signs(block, seed=SEED):
    return np.random.default_rng(seed).integers(0, 2, size=block).astype(np.float64) * 2 - 1

def _erfinv(y):
    # Exact scipy when present; else the oracle's Winitzki approx (~1e-3) so the
    # bracket is self-contained and never crashes on a missing-scipy runtime.
    try:
        from scipy.special import erfinv
        return erfinv(y)
    except Exception:
        a = 0.147
        ln = np.log(np.maximum(1 - y * y, 1e-300)); t = 2 / (np.pi * a) + ln / 2
        return np.sign(y) * np.sqrt(np.sqrt(t * t - ln / a) - t)

_LLOYD = {}
def _lloyd_max_gaussian(nlevels, iters=80, samples=400000, seed=SEED):
    if nlevels in _LLOYD: return _LLOYD[nlevels]
    rng = np.random.default_rng(seed)
    x = np.sort(rng.standard_normal(samples))
    p = (np.arange(nlevels) + 0.5) / nlevels
    lv = np.sqrt(2.0) * _erfinv(2.0 * p - 1.0)
    for _ in range(iters):
        b = (lv[:-1] + lv[1:]) / 2.0
        idx = np.searchsorted(b, x)
        sums = np.bincount(idx, weights=x, minlength=nlevels)
        cnts = np.bincount(idx, minlength=nlevels)
        new = np.where(cnts > 0, sums / np.maximum(cnts, 1), lv)
        if np.allclose(new, lv, atol=1e-8): lv = new; break
        lv = new
    _LLOYD[nlevels] = lv; return lv

def _scalar_q(y, levels):
    b = (levels[:-1] + levels[1:]) / 2.0
    return levels[np.searchsorted(b, y.ravel())].reshape(y.shape)

def qtip_bracket(W, store_bits=BITS, quality_bits=None, rht_block=RHT_BLOCK, seed=SEED):
    """RHT + Lloyd-Max scalar at the bracketed rate. quality_bits=store_bits is
    the LOWER bound (no trellis gain); store_bits+1 is the UPPER bound. Returns
    (recon, eff_bits, block_bytes). Identical to the oracle's qtip_quantize."""
    if quality_bits is None: quality_bits = store_bits
    levels = _lloyd_max_gaussian(1 << int(quality_bits))
    signs = _rht_signs(rht_block, seed)
    R, C = W.shape; nblk = C // rht_block
    recon = np.array(W, dtype=np.float64, copy=True)
    if nblk:
        seg = W[:, :nblk * rht_block].reshape(R, nblk, rht_block)
        rot = _fwht(seg * signs)
        sigma = rot.std(axis=-1, keepdims=True); sigma = np.where(sigma == 0, 1.0, sigma)
        qhat = _scalar_q(rot / sigma, levels) * sigma
        derot = _fwht(qhat) * signs
        recon[:, :nblk * rht_block] = derot.reshape(R, nblk * rht_block)
    eff_bits = store_bits + 16.0 / rht_block
    block_bytes = (store_bits * QK) / 8.0 + (16.0 / 8.0) * (QK / rht_block)
    return recon, eff_bits, block_bytes

def rel_rmse(recon, ref):
    recon = recon.ravel().astype(np.float64); ref = ref.ravel().astype(np.float64)
    d = np.linalg.norm(ref)
    return float(np.linalg.norm(recon - ref) / d) if d > 0 else float("nan")

print("codec bracket ready (RHT exact + Lloyd-Max {K,K+1}); block_bytes ~",
      f"{qtip_bracket(np.zeros((1, QK), np.float32))[2]:.0f} B/256-blk at {BITS} bits")


In [ ]:
# --- 3b. REAL upstream QTIP codec (Cornell-RelaxML/QTIP), fit from f16 --------
# Drive-RESUMABLE: a previously-computed QTIP HF model (or Hessians/quant ckpt)
# is restored from the Drive cache so a runtime restart never redoes the
# ~20-40 min codec. Every step is guarded; on any failure it falls back to the
# 3a bracket and the verdict downgrades to NEEDS-MEASUREMENT (it never
# fabricates a trellis number). Set USE_DRIVE_CACHE=False to skip Drive.
import shutil
HF_DIR = Path("/content/qwen3b-qtip-3bit-hf")
HESS = Path("/content/hessians_code"); CKPT = Path("/content/qwen3b-qtip-3bit")
results["real_qtip_ran"] = False

USE_DRIVE_CACHE = True
CACHE = None
if USE_DRIVE_CACHE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CACHE = Path("/content/drive/MyDrive/qtip_gate_cache"); CACHE.mkdir(parents=True, exist_ok=True)
        for art in (HF_DIR, CKPT, HESS):
            cached = CACHE / art.name
            if not art.exists() and cached.exists():
                (shutil.copytree if cached.is_dir() else shutil.copy2)(cached, art)
                print("restored from Drive cache:", art.name)
    except Exception as e:
        print("[Drive cache unavailable]", repr(e)[:160])

if HF_DIR.is_dir():
    results["real_qtip_ran"] = True
    print("QTIP HF model already present -> skipping codec:", HF_DIR)
elif RUN_REAL_QTIP:
    if not Path("QTIP").is_dir():
        subprocess.run(["git", "clone",
                        "https://github.com/Cornell-RelaxML/QTIP.git"], check=True)
    # Blackwell (sm_120) has no flash-attn wheel; make QTIP fall back to SDPA so
    # it doesn't ImportError / request flash_attention_2.
    subprocess.run(["bash", "-lc",
        "grep -rl flash_attention_2 QTIP 2>/dev/null | xargs -r sed -i 's/flash_attention_2/sdpa/g'; "
        "grep -rl 'import flash_attn' QTIP 2>/dev/null | xargs -r sed -i "
        "'s/^import flash_attn.*/flash_attn = None  # disabled on Blackwell/'"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "QTIP/requirements.txt"], check=False)
    # QTIP imports glog/primefac; the -r install can abort early on Blackwell
    # (flash-attn build) before reaching them -> install explicitly.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "glog", "primefac"], check=False)
    kbuild = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                             "QTIP/qtip-kernels"], capture_output=True, text=True)
    results["qtip_kernels_build_rc"] = kbuild.returncode
    if kbuild.returncode != 0:
        print("[warn] qtip-kernels build failed:", kbuild.stderr[-600:])
        print("[warn] -> using the 3a bracket; verdict downgrades to NEEDS-MEASUREMENT.")
    else:
        os.chdir("QTIP")
        def _qrun(mod, *a):
            r = subprocess.run([sys.executable, "-m", mod, *a],
                               capture_output=True, text=True)
            if r.returncode != 0:
                results.setdefault("qtip_errs", {})[mod] = (r.stderr or r.stdout)[-2500:]
                print(f"[{mod} FAILED rc={r.returncode}]\n", (r.stderr or r.stdout)[-2500:])
            return r.returncode
        # Hessians on the CODE calib corpus (skip if already present/restored).
        rc_h = 0 if HESS.exists() else _qrun("quantize_llama.input_hessian_llama",
            "--base_model", MODEL_ID, "--save_path", str(HESS),
            "--devset_size", "256", "--ctx_size", "1024")
        results["hessian_rc"] = rc_h
        rc_q = rc_hf = 1
        if rc_h == 0:
            # QTIP-3bit bitshift trellis (skip if the ckpt already exists/restored).
            rc_q = 0 if CKPT.exists() else _qrun("quantize_llama.quantize_finetune_llama",
                "--base_model", MODEL_ID, "--in_hess_path", str(HESS), "--save_path", str(CKPT),
                "--codebook", "bitshift", "--scale_override", "0.9", "--ft_epochs", "5",
                "--td_x", "16", "--td_y", "16", "--L", "16", "--K", str(BITS), "--V", "2",
                "--decode_mode", "quantlut_sym", "--tlut_bits", "9")
            results["quant_rc"] = rc_q
        if rc_q == 0:
            rc_hf = _qrun("quantize_llama.hfize_llama", "--quantized_path", str(CKPT),
                "--hf_output_path", str(HF_DIR))
            results["hfize_rc"] = rc_hf
        os.chdir("..")
        results["real_qtip_ran"] = (rc_h == 0 and rc_q == 0 and rc_hf == 0
                                    and HF_DIR.is_dir())
        # Bank the codec output to Drive so a restart resumes instantly.
        if results["real_qtip_ran"] and CACHE is not None:
            try:
                for art in (HF_DIR, CKPT, HESS):
                    if art.exists():
                        shutil.copytree(art, CACHE / art.name, dirs_exist_ok=True)
                print("saved QTIP artifacts to Drive cache:", CACHE)
            except Exception as e:
                print("[Drive save failed]", repr(e)[:160])
    print("real QTIP HF model ready:", results["real_qtip_ran"], "->", HF_DIR)
else:
    print("RUN_REAL_QTIP=False (VRAM<24GB). Using the 3a bracket only;",
          "verdict will be NEEDS-MEASUREMENT and name the real-QTIP rebuild gate.")


In [ ]:
# --- 4. LEG 1: weight-RMSE + the measured ~1.20-bit gap ----------------------
# Compare, per tensor, the RMSE-vs-f16 of (a) the gold Q4_K_M decode, (b) the
# QTIP bracket [lower, upper] at the ~3-bit budget, and -- if it ran -- (c) the
# REAL QTIP decode. The decisive scalar is bits_needed = log2(rmse_qtip /
# rmse_q4km): the proxy estimated ~1.20; GO needs the measured value <= 0.
from safetensors import safe_open
from huggingface_hub import snapshot_download
from gguf import GGUFReader, dequantize
import gc
import torch  # bf16-safe tensor load

# Representative tensors (mirror the oracle's SAMPLE_NAMES, HF naming).
HF_SAMPLE = [
    "model.layers.0.self_attn.q_proj.weight",
    "model.layers.0.mlp.gate_proj.weight",
    "model.layers.0.mlp.down_proj.weight",
    "model.layers.17.self_attn.o_proj.weight",
    "model.layers.17.mlp.up_proj.weight",
    "model.layers.35.self_attn.q_proj.weight",
    "model.layers.35.mlp.down_proj.weight",
]
GGUF_OF = {  # HF name -> gguf name
    "model.layers.0.self_attn.q_proj.weight": "blk.0.attn_q.weight",
    "model.layers.0.mlp.gate_proj.weight": "blk.0.ffn_gate.weight",
    "model.layers.0.mlp.down_proj.weight": "blk.0.ffn_down.weight",
    "model.layers.17.self_attn.o_proj.weight": "blk.17.attn_output.weight",
    "model.layers.17.mlp.up_proj.weight": "blk.17.ffn_up.weight",
    "model.layers.35.self_attn.q_proj.weight": "blk.35.attn_q.weight",
    "model.layers.35.mlp.down_proj.weight": "blk.35.ffn_down.weight",
}

# f16 weights from the HF snapshot (the fair quantizer input).
snap = snapshot_download(MODEL_ID, allow_patterns=["*.safetensors", "*.json"])
def _open_f16(name):
    for st in Path(snap).glob("*.safetensors"):
        with safe_open(str(st), framework="pt") as f:
            if name in f.keys():
                return f.get_tensor(name).to(torch.float32).cpu().numpy()
    return None

# Q4_K_M decoded weights from the gold GGUF.
q4reader = GGUFReader(Q4KM_GGUF)
q4by = {t.name: t for t in q4reader.tensors}
def _deq_q4km(gname):
    t = q4by.get(gname)
    if t is None: return None, None
    # Return FLAT dequant + dtype; the caller picks the (out,in) vs (in,out)
    # orientation that matches f16 (GGUF<->HF transpose ambiguity).
    return dequantize(np.array(t.data), t.tensor_type).astype(np.float32).ravel(), t.tensor_type.name

rmse_rows = []
for hf_name in HF_SAMPLE:
    Wf16 = _open_f16(hf_name)
    flat, disk = _deq_q4km(GGUF_OF[hf_name])
    if Wf16 is None or flat is None or flat.size != Wf16.size \
       or Wf16.shape[1] % QK:
        print("[skip]", hf_name); continue
    # Pick the layout matching f16 (a wrong GGUF<->HF transpose shows ~10x RMSE).
    _ca = flat.reshape(Wf16.shape)
    _cb = flat.reshape(Wf16.shape[::-1]).T
    Wq4 = _ca if rel_rmse(_ca, Wf16) <= rel_rmse(_cb, Wf16) else _cb
    rr_q4 = rel_rmse(Wq4, Wf16)
    lo, _, qtb = qtip_bracket(Wf16, store_bits=BITS, quality_bits=BITS)
    hi, _, _ = qtip_bracket(Wf16, store_bits=BITS, quality_bits=BITS + 1)
    rr_lo, rr_hi = rel_rmse(lo, Wf16), rel_rmse(hi, Wf16)
    row = dict(name=hf_name, disk=disk, shape=list(Wf16.shape),
               q4k_rmse=rr_q4, qtip_bracket_lower=rr_lo, qtip_bracket_upper=rr_hi,
               qtip_block_bytes=qtb, q4k_block_bytes=144.0)
    if results.get("real_qtip_ran"):
        # Real QTIP decoded weights (RHT+trellis already applied inside HF model).
        Wqt = None
        for st in Path(HF_DIR).glob("*.safetensors"):
            with safe_open(str(st), framework="pt") as f:
                if hf_name in f.keys():
                    Wqt = f.get_tensor(hf_name).to(torch.float32).cpu().numpy(); break
        if Wqt is not None and Wqt.shape == Wf16.shape:
            row["qtip_real_rmse"] = rel_rmse(Wqt, Wf16)
        del Wqt
    rmse_rows.append(row)
    print(f"  {hf_name:42s} Q4K={rr_q4:.4f}  QTIP[lo,hi]=[{rr_lo:.4f},{rr_hi:.4f}]"
          + (f"  REAL={row.get('qtip_real_rmse', float('nan')):.4f}"
             if 'qtip_real_rmse' in row else ""))
    del Wf16, Wq4, lo, hi; gc.collect()

results["rmse_per_tensor"] = rmse_rows
med = lambda k: float(np.median([r[k] for r in rmse_rows if k in r])) if rmse_rows else float("nan")
m_q4 = med("q4k_rmse"); m_lo = med("qtip_bracket_lower"); m_hi = med("qtip_bracket_upper")
results["median_q4k_rmse"] = m_q4
results["median_qtip_bracket"] = [m_lo, m_hi]
bn = lambda r: float(np.log2(r / m_q4)) if (r > 0 and m_q4 > 0) else float("nan")
results["bits_needed_bracket"] = [bn(m_lo), bn(m_hi)]   # log2(qtip/q4k); <=0 is GO
print(f"\n[LEG 1] median RMSE  Q4_K_M={m_q4:.4f}  QTIP bracket=[{m_lo:.4f},{m_hi:.4f}]")
print(f"        bracket bits_needed = log2(QTIP/Q4K) = "
      f"[{bn(m_lo):+.2f}, {bn(m_hi):+.2f}]  (proxy est ~{PROXY_BITS_NEEDED}; GO needs <= {GATE_BITS_NEEDED})")
if results.get("real_qtip_ran") and all("qtip_real_rmse" in r for r in rmse_rows):
    m_real = med("qtip_real_rmse"); results["median_qtip_real_rmse"] = m_real
    results["bits_needed_real"] = bn(m_real)
    print(f"        REAL QTIP median RMSE={m_real:.4f}  bits_needed={bn(m_real):+.2f}"
          f"  -> {'<= 0 PASS' if bn(m_real) <= GATE_BITS_NEEDED else '> 0 FAIL'} (measured)")

In [ ]:
# --- 5. LEG 2 (DECISIVE): logit-cosine / KL / argmax vs Q4_K_M on code -------
# Forward-pass f16, gold Q4_K_M, and (if it ran) REAL QTIP on held-out CODE
# tokens; compare next-token logits. This is the W4A8 metric class -- the
# decisive functional leg. GO requires QTIP to be at least as good as the
# Q4_K_M incumbent it would replace. Q4_K_M logits come from llama.cpp via its
# python bindings if available, else from a transformers GGUF load.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
code_eval = Path(EVAL).read_text(encoding="utf-8", errors="replace")
N_TOK = 1024
ids = tok(code_eval, return_tensors="pt", add_special_tokens=False).input_ids[:, :N_TOK]

@torch.inference_mode()
def next_logits(model):
    out = model(ids.to(model.device)).logits[0].float().cpu().numpy()  # (T, V)
    return out[:-1]                                                     # predict t+1

def logit_cosine(a, b):
    a = a.astype(np.float64); b = b.astype(np.float64)
    num = np.sum(a * b, -1); den = np.linalg.norm(a, axis=-1) * np.linalg.norm(b, axis=-1)
    return float(np.mean(num / np.maximum(den, 1e-12)))
def _softmax(x):
    x = x - x.max(-1, keepdims=True); e = np.exp(x); return e / e.sum(-1, keepdims=True)
def kl_div(p_logits, q_logits):
    p = _softmax(p_logits.astype(np.float64)); q = _softmax(q_logits.astype(np.float64))
    return float(np.mean(np.sum(p * (np.log(p + 1e-12) - np.log(q + 1e-12)), -1)))
def argmax_agree(ref, other):
    return float(np.mean(np.argmax(ref, -1) == np.argmax(other, -1)))

# f16 reference logits.
m_f16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                             device_map="cuda", trust_remote_code=True)
L_f16 = next_logits(m_f16); del m_f16; torch.cuda.empty_cache()

# Gold Q4_K_M logits (llama-cpp-python is the faithful GGUF forward path).
L_q4 = None
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "llama-cpp-python"], check=False)
    from llama_cpp import Llama
    llm = Llama(model_path=Q4KM_GGUF, n_ctx=N_TOK + 8, logits_all=True,
                n_gpu_layers=-1, verbose=False)
    toklist = ids[0].tolist()
    llm.eval(toklist)
    L_q4 = np.array([llm.scores[i] for i in range(len(toklist) - 1)], dtype=np.float32)
    # Align vocab width with HF logits (llama.cpp may pad to n_vocab).
    V = min(L_f16.shape[1], L_q4.shape[1])
    L_q4 = L_q4[:L_f16.shape[0], :V]; L_f16c = L_f16[:, :V]
    del llm
except Exception as e:
    print("[warn] llama-cpp-python logits unavailable:", repr(e)[:200])
    print("[warn] falling back to transformers GGUF load for Q4_K_M logits.")
    m_q4 = AutoModelForCausalLM.from_pretrained(
        "/content", gguf_file=os.path.basename(Q4KM_GGUF),
        torch_dtype=torch.float16, device_map="cuda")
    L_q4 = next_logits(m_q4); del m_q4; torch.cuda.empty_cache()
    V = min(L_f16.shape[1], L_q4.shape[1])
    L_q4 = L_q4[:, :V]; L_f16c = L_f16[:, :V]

logit = {"cos_q4k": logit_cosine(L_q4, L_f16c), "kl_q4k": kl_div(L_f16c, L_q4),
         "argmax_q4k": argmax_agree(L_f16c, L_q4)}

# REAL QTIP logits (only if the codec actually ran).
if results.get("real_qtip_ran"):
    m_qt = AutoModelForCausalLM.from_pretrained(str(HF_DIR), torch_dtype=torch.float16,
                                                device_map="cuda")
    L_qt = next_logits(m_qt)[:, :V]; del m_qt; torch.cuda.empty_cache()
    logit.update(cos_qtip=logit_cosine(L_qt, L_f16c), kl_qtip=kl_div(L_f16c, L_qt),
                 argmax_qtip=argmax_agree(L_f16c, L_qt))
results["logit"] = logit
# Free the large fp32 logit arrays before LEG 3 reloads the models (RAM).
import gc as _gc
for _n in ("L_f16", "L_f16c", "L_q4", "L_qt"):
    globals().pop(_n, None)
_gc.collect()
try:
    torch.cuda.empty_cache()
except Exception:
    pass
print("[LEG 2] vs f16 on code:")
print(f"  Q4_K_M : cos={logit['cos_q4k']:.5f}  KL={logit['kl_q4k']:.5f}  argmax={logit['argmax_q4k']:.4f}")
if "cos_qtip" in logit:
    print(f"  QTIP   : cos={logit['cos_qtip']:.5f}  KL={logit['kl_qtip']:.5f}  argmax={logit['argmax_qtip']:.4f}  (measured)")
    print(f"  GO floor: QTIP cos>={logit['cos_q4k']:.5f}, argmax>={logit['argmax_q4k']:.4f}, KL<={logit['kl_q4k']:.5f}")
else:
    print("  QTIP   : real codec did not run -> logit leg is NEEDS-MEASUREMENT.")

In [ ]:
# --- 6. LEG 3 (corroborating): code PPL (f16 / gold Q4_K_M / real QTIP) -------
# llama.cpp gives the gold Q4_K_M PPL with the same tool the local oracle used
# (so it is directly comparable). f16 and QTIP PPL come from transformers.
NGL = ["-ngl", "99"]
def llama_ppl(gguf):
    r = subprocess.run([f"{BIN}/llama-perplexity", "-m", gguf, "-f", EVAL,
                        "--chunks", "50", *NGL], capture_output=True, text=True)
    m = re.findall(r"Final estimate:\s*PPL\s*=\s*([0-9.]+)", r.stderr + r.stdout)
    return float(m[-1]) if m else None

@torch.inference_mode()
def hf_ppl(model, text, max_len=2048, stride=512, max_eval=16384):
    iid = tok(text, return_tensors="pt", add_special_tokens=False).input_ids[:, :max_eval].to(model.device)
    nlls, n, prev = [], 0, 0
    for beg in range(0, iid.size(1), stride):
        end = min(beg + max_len, iid.size(1)); trg = end - prev
        inp = iid[:, beg:end]; tgt = inp.clone(); tgt[:, :-trg] = -100
        nlls.append(model(inp, labels=tgt).loss.float() * trg); n += trg; prev = end
        if end == iid.size(1): break
    return float(torch.exp(torch.stack(nlls).sum() / n))

results["ppl_q4k"] = llama_ppl(Q4KM_GGUF)
m_f16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                             device_map="cuda", trust_remote_code=True)
results["ppl_f16"] = hf_ppl(m_f16, code_eval); del m_f16; torch.cuda.empty_cache()
if results.get("real_qtip_ran"):
    m_qt = AutoModelForCausalLM.from_pretrained(str(HF_DIR), torch_dtype=torch.float16,
                                                device_map="cuda")
    results["ppl_qtip"] = hf_ppl(m_qt, code_eval); del m_qt; torch.cuda.empty_cache()
print(f"[LEG 3] code PPL  f16={results['ppl_f16']}  Q4_K_M={results['ppl_q4k']}"
      + (f"  QTIP={results['ppl_qtip']} (measured)" if 'ppl_qtip' in results else
         "  QTIP=NEEDS-MEASUREMENT"))

In [ ]:
# --- 7. VERDICT: the three legs against the baked-in lines -------------------
# GO requires the REAL codec to clear ALL of: weight-RMSE bits_needed <= 0,
# logit cos/argmax >= Q4_K_M & KL <= Q4_K_M, and PPL <= 1.10x f16 (and <= Q4K).
# If the real codec did NOT run, the verdict is NEEDS-MEASUREMENT (a CPU/bracket
# result cannot legitimately KILL QTIP -- Type-2 error) and names the gate.
lg = results.get("logit", {})
leg1 = leg2 = leg3 = None
if results.get("real_qtip_ran") and "bits_needed_real" in results:
    leg1 = results["bits_needed_real"] <= GATE_BITS_NEEDED
if "cos_qtip" in lg:
    leg2 = (lg["cos_qtip"] >= lg["cos_q4k"] and lg["argmax_qtip"] >= lg["argmax_q4k"]
            and lg["kl_qtip"] <= lg["kl_q4k"])
if results.get("ppl_qtip") and results.get("ppl_f16") and results.get("ppl_q4k"):
    leg3 = (results["ppl_qtip"] <= GATE_PPL_RATIO * results["ppl_f16"]
            and results["ppl_qtip"] <= results["ppl_q4k"])
results["legs"] = {"weight_rmse_bits_needed_le0": leg1,
                   "logit_ge_q4k": leg2, "ppl_le_gate": leg3}

if results.get("real_qtip_ran") and None not in (leg1, leg2, leg3):
    results["verdict"] = "GO" if (leg1 and leg2 and leg3) else "NO-GO"
    if results["verdict"] == "NO-GO":
        # Kill-protocol: a real-codec NO-GO on code at 3 bits is a measured
        # property -> quality Type-1. Records to dead_levers in the repo.
        results["kill_type"] = "Type-1 (quality): real QTIP-3.0 from f16 does not match Q4_K_M on code"
else:
    results["verdict"] = "NEEDS-MEASUREMENT"
    results["decisive_gate"] = ("re-run with RUN_REAL_QTIP=True on an L4/A100 so the "
        "upstream QTIP RHT+trellis codec builds and HF-izes; the bracket alone "
        "cannot legitimately kill or pass QTIP (Type-2 error to try).")

print("=" * 64)
print(f"QTIP-{BITS}bit (from f16) vs Q4_K_M on CODE")
print(f"  LEG 1 weight-RMSE bits_needed<=0 : {leg1}"
      + (f"  (measured {results['bits_needed_real']:+.2f})" if 'bits_needed_real' in results
         else f"  (bracket [{results['bits_needed_bracket'][0]:+.2f},{results['bits_needed_bracket'][1]:+.2f}], proxy ~{PROXY_BITS_NEEDED})"))
print(f"  LEG 2 logit cos/argmax>=Q4K,KL<= : {leg2}")
print(f"  LEG 3 PPL<=1.10x f16 and <=Q4K   : {leg3}")
print(f"  VERDICT: {results['verdict']}")
if results.get("decisive_gate"): print("  GATE:", results["decisive_gate"])
if results.get("kill_type"): print("  KILL:", results["kill_type"])
print("=" * 64)

Path("qtip_3bit_results.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))
try:
    from google.colab import files
    files.download("qtip_3bit_results.json")
except Exception:
    pass

## What a GO / NO-GO / NEEDS-MEASUREMENT means here

- **GO** (real codec ran, all three legs clear): QTIP-3.0 from f16 matches-or-
  beats Q4_K_M on code at ~96–104 B/block (−41 to −45% bytes). This clears the
  **quality** gate (design §5.1). It does NOT yet clear the **decode-cost** gate
  — the M3 trellis kernel must still be proven bandwidth-bound (design §5.2 /
  §6.3), which is the §5.0 by-proxy `q3k_bytecut_bench` read first, then a hand-
  written `gemm_qtip_trellis_v1`. A GO here only earns that kernel work.
- **NO-GO** (real codec ran, a leg failed): records a **quality Type-1** kill to
  `reports/dead_levers.md` — a 3B at 3 bits via this codec is not good enough on
  the product workload (a measured property, not effort). Closes the byte-cut
  axis together with the §5.0/§6 speed read. Never re-tested on vibes.
- **NEEDS-MEASUREMENT** (only the bracket ran): the assigned GPU could not build
  the QTIP CUDA kernels, or VRAM<24GB. The bracket [lower, upper] is reported
  but a bracket/CPU result **cannot legitimately kill or pass** QTIP (that is the
  Type-2 error CLAUDE.md forbids). Re-run on an L4/A100 with `RUN_REAL_QTIP=True`.

## M3 integration boundary (unchanged)

A QTIP quality GO is still only a quality result. Shipping it requires a native
trellis-decode GEMV in Metal (`gemm_qtip_trellis_v1`, design §6.3) plus a GGUF-
adjacent loader format, and the encoder must emit **lane-independent sub-blocks**
so 32 simdgroup lanes decode in parallel (the Apple co-design constraint, §7).
Do not spend that kernel effort until this gate is GO and the §5.0 by-proxy
`q3k_bytecut_bench` read says a Q3-class format can be bandwidth-bound on the M3.